# Postprocess into rasters and vector datasets




In [1]:
import geopandas as gpd
import geoutils as gu
import numpy as np
import xdem
from osgeo import gdal, ogr, osr
import rasterstats
from osgeo_utils import gdal_calc


import subkart



In [2]:
nodata = 255
crs = "EPSG:25833"

In [3]:
classifier = subkart.utils.load_classifier()

## Post processing

In [4]:
fname = subkart.utils.to_filename(
    f"nisjedata-substrat-{classifier.__class__.__name__.lower()}", "norge", "latest", crs.split(":")[1]
)
predict_file = f"{fname}_unmapped.tif"
prob_file = f"{fname}_3band_probability.tif"

In [7]:
gdal.UseExceptions()

prediction_files = [f"{r}_prediction.tif" for r in subkart.sources.REGIONS]
probability_files = [f"{r}_probability.tif" for r in subkart.sources.REGIONS]

subkart.utils.merge_rasters(prediction_files, predict_file, nodata=nodata)
subkart.utils.merge_rasters(probability_files, prob_file, nodata=-9999)


# Create processed prediction raster:

Remap class 1 (blanding) → highest-probability class (0=løsbunn or 2=fastbunn) using gdal_calc

In [5]:
predict_file_remapped = f"{fname}.tif"

In [9]:

pred_ds = gdal.Open(predict_file)
prob_ds = gdal.Open(prob_file)

driver = gdal.GetDriverByName("GTiff")
out_ds = driver.Create(
    predict_file_remapped,
    pred_ds.RasterXSize,
    pred_ds.RasterYSize,
    1,
    gdal.GDT_Byte,
    options=["COMPRESS=DEFLATE", "TILED=YES", "BLOCKXSIZE=256", "BLOCKYSIZE=256", "BIGTIFF=IF_SAFER"],
)
out_ds.SetGeoTransform(pred_ds.GetGeoTransform())
out_ds.SetProjection(pred_ds.GetProjection())

out_band = out_ds.GetRasterBand(1)
out_band.SetNoDataValue(nodata)
pred_band = pred_ds.GetRasterBand(1)
prob_band1 = prob_ds.GetRasterBand(1)
prob_band3 = prob_ds.GetRasterBand(3)

block_size = 256
xsize, ysize = pred_ds.RasterXSize, pred_ds.RasterYSize
for y in range(0, ysize, block_size):
    ny = min(block_size, ysize - y)
    for x in range(0, xsize, block_size):
        nx = min(block_size, xsize - x)
        A = pred_band.ReadAsArray(x, y, nx, ny)
        B = prob_band1.ReadAsArray(x, y, nx, ny)
        C = prob_band3.ReadAsArray(x, y, nx, ny)
        result = np.where(A == 1, np.where(B >= C, np.uint8(0), np.uint8(2)), A)
        out_band.WriteArray(result, x, y)

out_ds.FlushCache()
out_ds = None
pred_ds = None
prob_ds = None


## Create processed 1-band probability raster from the 3-band source

* class 0 (løsbunn, incl. remapped/sieved blanding) band 1 = P(class=0)
* class 2 (fastbunn)                            band 3 = P(class=2)

In [7]:
prob_file_processed = f"{fname}_probability.tif"

In [ ]:
pred_ds = gdal.Open(predict_file_remapped)
prob_ds = gdal.Open(prob_file)

driver = gdal.GetDriverByName("GTiff")
out_ds = driver.Create(
    prob_file_processed,
    pred_ds.RasterXSize,
    pred_ds.RasterYSize,
    1,
    gdal.GDT_Float32,
    options=["COMPRESS=DEFLATE", "TILED=YES", "BLOCKXSIZE=256", "BLOCKYSIZE=256", "BIGTIFF=IF_SAFER"],
)
out_ds.SetGeoTransform(pred_ds.GetGeoTransform())
out_ds.SetProjection(pred_ds.GetProjection())

out_band = out_ds.GetRasterBand(1)
out_band.SetNoDataValue(-9999)
pred_band = pred_ds.GetRasterBand(1)
prob_band1 = prob_ds.GetRasterBand(1)
prob_band3 = prob_ds.GetRasterBand(3)

block_size = 256
xsize, ysize = pred_ds.RasterXSize, pred_ds.RasterYSize
for y in range(0, ysize, block_size):
    ny = min(block_size, ysize - y)
    for x in range(0, xsize, block_size):
        nx = min(block_size, xsize - x)
        A = pred_band.ReadAsArray(x, y, nx, ny)
        B = prob_band1.ReadAsArray(x, y, nx, ny).astype(np.float32)
        C = prob_band3.ReadAsArray(x, y, nx, ny).astype(np.float32)
        denom = B + C
        result = np.where(A == nodata, np.float32(-9999), np.where(A == 2, C / denom, B / denom))
        out_band.WriteArray(result, x, y)

out_ds.FlushCache()
out_ds = None
pred_ds = None
prob_ds = None


## Vectorize processed prediction raster

In [8]:
subkart.vectorize.with_gdal(
    predict_file_remapped, "polygons_processed.gpkg", epsg_code=int(crs.split(":")[1])
)

gdf = gpd.read_file("polygons_processed.gpkg").explode()

reverse_map = {v: k for k, v in subkart.labelling.BUNNTYPE_MAPPING.items()}
gdf["BunnType"] = gdf["DN"].map(reverse_map)

# Compute mean probability per polygon from the processed 1-band probability raster
stats = rasterstats.zonal_stats(
    gdf,
    prob_file_processed,
    stats=["mean"],
    nodata=-9999,
)
gdf["sannsynlighet"] = [s["mean"]*100 for s in stats]

gdf.to_file(f"{fname}.gpkg", driver="GPKG", layer="bunntyper")
gdf.to_parquet(f"{fname}.geo.parquet", compression="snappy")


Polygons saved to polygons_processed.gpkg


In [9]:
subkart.utils.to_postgis(gdf, fname)

Table nisjedata_substrat_xgbclassifier_norge_latest uploaded to PostGIS.
